In [3]:
import ollama

_instalados = [m["model"] for m in ollama.list()["models"]] #modelos instalados
MODELO_EMBED = next((t for t in ["embddinggemma", "embeddinggemma:300m", "embeddinggemma:latest"] if t in _instalados), "embeddinggemma") 

MODELO_CHAT = "gemma4:e2b" #el modelo que redacta las respuestas, se puede cambiar por otro modelo de ollama

print("Embedding:", MODELO_EMBED)
print("Chat:", MODELO_CHAT)
print("Instalados:", MODELO_EMBED in _instalados, MODELO_CHAT in _instalados)

Embedding: embeddinggemma:300m
Chat: gemma4:e2b
Instalados: True True


In [4]:
def preguntar(prompt, system=None):
    """Manda un mensaje al modelo de CHAT y devuelve solo el texto de la respuesta.
    prompt :  la pregunta o instrucción del usuario (rol "user").
    system : instrucciones de comportamiento opcionales para el modelo (rol "system").
        Aquí añadiremos más adelante las reglas del RAG.
    """

    # ollama espera una LISTA de mensajes con su rol (igual que la API de OPENAI)
    mensajes = []
    if system:
        mensajes.append({"role": "system", "content": system})
    mensajes.append({"role": "user", "content": prompt})

    opciones = {"temperature": 0} #opciones de generación, se pueden cambiar
    if "qwen" in MODELO_CHAT:
        opciones["think"] = False #qwen3 tiene una opción para "pensar" antes de responder, pero no siempre da buenos resultados

    r = ollama.chat(model=MODELO_CHAT, messages=mensajes, options=opciones)
    return r["message"]["content"].strip() #nos quedamos solo con el texto sin metadatos

print(preguntar("Di 'listo'"))

¿Qué quieres que te diga?

(What do you want me to say to you?)


In [5]:
pregunta_de_prueba = "¿Cuántos días de vacaciones me corresponden al año en Lumetra?"

print(preguntar(pregunta_de_prueba))

Como modelo de lenguaje, no tengo acceso a tu información personal, datos de nómina, ni a las políticas internas de vacaciones de ninguna empresa específica como Lumetra.

Para saber cuántos días de vacaciones te corresponden, debes consultar:

1. **Tu departamento de Recursos Humanos (RR.HH.).**
2. **Tu contrato de trabajo.**
3. **El portal interno de la empresa (si tienen uno).**

Ellos son la única fuente oficial para esta información.


In [6]:
r = ollama.embed(model=MODELO_EMBED, input="El gato duerme en el sofá")
vector = r["embeddings"][0] #el resultado es una lista de listas, nos quedamos con la primera

print(f"Dimensión del vector: {len(vector)}") 
print(f"Primeras 5 componentes: {vector[:5]}")

Dimensión del vector: 768
Primeras 5 componentes: [-0.13204777, 0.03927413, 0.04070033, -0.022822458, 0.013374283]


In [7]:
import numpy as np

def coseno(a, b):
    """Calcula la similitud coseno entre dos vectores a y b."""
    a = np.array(a)
    b = np.array(b)
    # producto escalar (a @ b) dividido por el producto de las normas
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

frases = [
    "El gato duerme en el sofá",
    "Un felino descansa en el sillón",
    "La factura vence el 30 de junio",
    "El pago debe realizarse antes de fin de mes"
]

vecs = ollama.embed(model=MODELO_EMBED, input=frases)["embeddings"] #vectorizamos todas las frases


# comparamos todas las parejas posibles de frases
for i in range(len(frases)):
    for j in range(i+1, len(frases)):       
        print(f"{coseno(vecs[i], vecs[j]):.3f}  |  '{frases[i]}'  <->  '{frases[j]}'")
 

0.769  |  'El gato duerme en el sofá'  <->  'Un felino descansa en el sillón'
0.272  |  'El gato duerme en el sofá'  <->  'La factura vence el 30 de junio'
0.266  |  'El gato duerme en el sofá'  <->  'El pago debe realizarse antes de fin de mes'
0.228  |  'Un felino descansa en el sillón'  <->  'La factura vence el 30 de junio'
0.243  |  'Un felino descansa en el sillón'  <->  'El pago debe realizarse antes de fin de mes'
0.599  |  'La factura vence el 30 de junio'  <->  'El pago debe realizarse antes de fin de mes'


In [8]:
# Embeddinggemma se entrenó con "prefijos" que le indican qué papel juega el texto.
# Usar el prefijoi correcto mejora el emparejamiento pregunta <-> documento.

def embed_documentos(textos):
    """Vectoriza una LISTA de textos que vamos a guardar en la base (documentos)."""
    entrada = [f"title: none | text: {t}" for t in textos] #añadimos el prefijo (de documentos) "title: none | text:" a cada texto
    return ollama.embed(model=MODELO_EMBED, input=entrada)["embeddings"]

def embed_consulta(texto):
    """Vectoriza UNA pregunta que vamos a BUSCAR (consulta)."""
    entrada = f"task: search result | query: {texto}" #añadimos el prefijo (de consulta) "query:" a la pregunta
    return ollama.embed(model=MODELO_EMBED, input=entrada)["embeddings"][0] #devuelve un solo vector

 

In [9]:
comentarios = [
    "Me han cobrado dos veces la suscripción este mes",
    "La app va lentísima desde la última actualización",
    "No consigo restablecer mi contraseña, el correo no llega",
    "El cargo de mi tarjeta no corresponde con la tarifa contratada",
    "Me encanta el nuevo diseño del panel de control, ¡muy intuitivo!",
    "Quiero darme de baja y nadie me responde"
]

def buscar (consulta, textos, k=3):
    """Busca los k textos más relevantes para la consulta dada, usando similitud coseno.
    Es un TAG en miniatura: vectorizar todo -> comparar -> ordenar -> quedarnos con los mejores.    
    """
    vec_consulta = embed_consulta(consulta) #vectorizamos la consulta
    vecs_textos = embed_documentos(textos) #vectorizamos los textos (documentos)

    #calculamos la similitud coseno entre la consulta y cada texto
    puntuaciones = [coseno(vec_consulta, v) for v in vecs_textos]

    orden = sorted(range(len(textos)), key=lambda i: puntuaciones[i], reverse=True) #ordenamos los índices de los textos por su puntuación
    return [(textos[i], puntuaciones[i]) for i in orden[:k]] #devolvemos los k mejores textos con su puntuación 


for texto, p in buscar("problemas para pagar", comentarios):
    print(f"{p:.3f}  |  {texto}")
 

0.286  |  Me han cobrado dos veces la suscripción este mes
0.279  |  El cargo de mi tarjeta no corresponde con la tarifa contratada
0.231  |  No consigo restablecer mi contraseña, el correo no llega


In [10]:
from pathlib import Path

# Cargar todos los .txt de la carpeta datos/ en un diccionario {nombre: contenido}
docs = {}
for ruta in sorted(Path("datos").glob("*.txt")):
    docs[ruta.name] = ruta.read_text(encoding="utf-8")

for nombre, texto in docs.items():
    print(f"{nombre}: {len(texto)} caracteres")    

faq_soporte.txt: 1171 caracteres
manual_producto.txt: 1192 caracteres
onboarding.txt: 1142 caracteres
politica_teletrabajo.txt: 1172 caracteres


In [11]:
# Troceamos cada documento en CHUNKS

chunks, metadatos = [], []
for nombre, texto in docs.items():
    for parrafo in texto.split("\n\n"): #los párrados por una línea en blanco
        parrafo = parrafo.strip() #quitamos espacios al principio y al final
        if len(parrafo) > 40: #si el párrafo tiene más de 40 caracteres, lo guardamos como chunk
            chunks.append(parrafo)
            metadatos.append({"fuente": nombre}) #guardamos el nombre del documento como metadato


print(f"{len(chunks)} chunks generados")   
print("Ejemplo de chunk:", chunks[0][:120],"...")         

18 chunks generados
Ejemplo de chunk: PREGUNTAS FRECUENTES DE SOPORTE — LUMETRA INSIGHT ...


In [12]:
import chromadb

# PersistentClient: los datos se guardan en disco y persisten entre ejecuciones
cliente = chromadb.PersistentClient(path="chroma_lumetra") 

# Una "colección" es como una tabla de base de datos donde guardaremos nuestros chunks y sus vectores
coleccion =  cliente.get_or_create_collection(
    "lumetras", metadata={"hnsw:space": "cosine"}
)


# upsert: inserta o actualiza un vector con su id, sus metadatos y su chunk de texto original
# Guardamos 4 cosas por chunk: id, texto, vector y metadato

coleccion.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))], #id único para cada chunk
    documents=chunks, #el texto original del chunk
    embeddings=embed_documentos(chunks), #el vector del chunk
    metadatas=metadatos #los metadatos (en este caso, la fuente)
)

print(f"Chunks guardados en la colección: {coleccion.count()}")

Chunks guardados en la colección: 18


In [13]:
def recuperar(pregunta, k=4):
    """Función de recuperación que busca los k chunks más relevantes para la pregunta dada.

     Usamos k=4 (y no 1) porque el chunk con el dato exacto no siempre es el #1 en la lista de resultados,
     a veces el modelo lo considera relevante pero no el más relevante.
     De esta forma, aunque el chunk con la respuesta exacta no sea el #1,
     al menos estará entre los 4 primeros y el modelo de CHAT podrá usarlo para redactar la respuesta.

    """
    #query_embeddings espera una lista de vectores de consulta, pero nosotros solo tenemos una pregunta, así que le pasamos una lista con un solo vector
    res = coleccion.query(query_embeddings=[embed_consulta(pregunta)], n_results=k)   
    #chroma devuelve listas de listas, nos quedamos con la primera (y única) consulta y hacemos zip para juntar cada chunk con su fuente 
    return list(zip(res["documents"][0], [m["fuente"] for m in res["metadatas"][0]])) 

for texto, fuente in recuperar("¿Cuántos días de vacaciones me corresponden?"):
    print(f"{fuente}: {texto[:110]}...")

politica_teletrabajo.txt: Durante julio y agosto se puede solicitar teletrabajo total (5 días remotos por semana). La solicitud se prese...
politica_teletrabajo.txt: Lumetra funciona con un modelo híbrido 3+2: tres días de teletrabajo y dos días presenciales por semana. Los d...
onboarding.txt: Cada nueva incorporación tiene asignado un buddy durante las primeras 6 semanas: una persona del equipo que re...
onboarding.txt: Las cuentas de correo, Slack y Atlas RRHH se activan el primer día. La firma de correo corporativa se configur...


In [19]:
for texto, fuente in recuperar("¿me pagan algo por currar desde casa?"):
    print(f"{fuente}: {texto[:110]}...")

politica_teletrabajo.txt: Toda persona en modalidad híbrida recibe una ayuda de 35 euros mensuales en nómina en concepto de gastos de te...
politica_teletrabajo.txt: Durante julio y agosto se puede solicitar teletrabajo total (5 días remotos por semana). La solicitud se prese...
politica_teletrabajo.txt: El equipamiento adicional para el puesto remoto (silla ergonómica y monitor externo) se solicita una única vez...
manual_producto.txt: El acuerdo de nivel de servicio (SLA) de los planes Pro y Enterprise garantiza una disponibilidad mensual del ...


Ejercicio

1. Probad 3 preguntas sobre Lumetra, ¿ sale el chunk correcto en la respuesta?
2. Construir una pregunta con sinonimos
3. Intented encontrar una pregunta donde el top 1 de chunk que sale este equivocado

In [21]:
# 1. Probad 3 preguntas sobre Lumetra, ¿ sale el chunk correcto en la respuesta?
preguntas = [
    "¿Qué es Lumetra?",
    "¿Qué servicios ofrece Lumetra?",
    "¿Cuál es el objetivo principal de Lumetra?"
]

for pregunta in preguntas:
    print("\n" + "="*80)
    print("PREGUNTA:", pregunta)

    resultados = recuperar(pregunta)

    for i, (texto, fuente) in enumerate(resultados, start=1):
        print(f"\nChunk {i}")
        print(f"Fuente: {fuente}")
        print(texto[:500])


PREGUNTA: ¿Qué es Lumetra?

Chunk 1
Fuente: manual_producto.txt
Lumetra Insight es la plataforma de cuadros de mando (dashboards) de Lumetra. Se conecta a PostgreSQL, BigQuery y hojas de cálculo de Google, y refresca los datos automáticamente cada 15 minutos en todos los planes. Los dashboards se comparten por enlace o se incrustan en webs internas.

Chunk 2
Fuente: politica_teletrabajo.txt
Lumetra funciona con un modelo híbrido 3+2: tres días de teletrabajo y dos días presenciales por semana. Los días presenciales obligatorios son martes y jueves, en la oficina de Madrid (Calle del Dato, 7, planta 3). Los equipos pueden pactar un tercer día presencial puntual para talleres o cierres de proyecto.

Chunk 3
Fuente: manual_producto.txt
El acuerdo de nivel de servicio (SLA) de los planes Pro y Enterprise garantiza una disponibilidad mensual del 99,9%. Si en un mes no se alcanza, Lumetra descuenta automáticamente el 10% de la factura del mes siguiente. El plan Starter no incluye SLA.

Chun

In [22]:
# 2. Pregunta con sinónimos
for texto, fuente in recuperar(
    "¿Qué soluciones proporciona Lumetra a sus clientes?"
):
    print(f"{fuente}: {texto[:300]}")

manual_producto.txt: El acuerdo de nivel de servicio (SLA) de los planes Pro y Enterprise garantiza una disponibilidad mensual del 99,9%. Si en un mes no se alcanza, Lumetra descuenta automáticamente el 10% de la factura del mes siguiente. El plan Starter no incluye SLA.
manual_producto.txt: Lumetra Insight es la plataforma de cuadros de mando (dashboards) de Lumetra. Se conecta a PostgreSQL, BigQuery y hojas de cálculo de Google, y refresca los datos automáticamente cada 15 minutos en todos los planes. Los dashboards se comparten por enlace o se incrustan en webs internas.
politica_teletrabajo.txt: Lumetra funciona con un modelo híbrido 3+2: tres días de teletrabajo y dos días presenciales por semana. Los días presenciales obligatorios son martes y jueves, en la oficina de Madrid (Calle del Dato, 7, planta 3). Los equipos pueden pactar un tercer día presencial puntual para talleres o cierres d
faq_soporte.txt: El horario de soporte es de lunes a viernes de 8:00 a 20:00 (hora de Madrid

In [23]:
# 3. Intented encontrar una pregunta donde el top 1 de chunk que sale este equivocado
preguntas_dificiles = [
    "¿Qué ventajas tiene?",
    "¿Cómo funciona?",
    "¿Qué ofrece la empresa?",
    "¿Me pagan algo por trabajar desde casa?"
]

for pregunta in preguntas_dificiles:
    print("\n" + "="*80)
    print("PREGUNTA:", pregunta)

    for texto, fuente in recuperar(pregunta):
        print(f"{fuente}: {texto[:200]}")


PREGUNTA: ¿Qué ventajas tiene?
politica_teletrabajo.txt: Toda persona en modalidad híbrida recibe una ayuda de 35 euros mensuales en nómina en concepto de gastos de teletrabajo. Los requisitos técnicos mínimos son una conexión de fibra de al menos 100 Mb y 
manual_producto.txt: El acuerdo de nivel de servicio (SLA) de los planes Pro y Enterprise garantiza una disponibilidad mensual del 99,9%. Si en un mes no se alcanza, Lumetra descuenta automáticamente el 10% de la factura 
manual_producto.txt: Planes y precios: el plan Starter cuesta 49 euros al mes e incluye 5 usuarios y un máximo de 50 dashboards. El plan Pro cuesta 199 euros al mes e incluye 25 usuarios, dashboards ilimitados y acceso a 
manual_producto.txt: Lumetra Insight es la plataforma de cuadros de mando (dashboards) de Lumetra. Se conecta a PostgreSQL, BigQuery y hojas de cálculo de Google, y refresca los datos automáticamente cada 15 minutos en to

PREGUNTA: ¿Cómo funciona?
politica_teletrabajo.txt: Lumetra funciona con u

In [24]:
SYSTEM_RAG ="""Eres el asistente interno de Lumetra.
Responde SOLO con la información del CONTEXTO.
Si la respuesta no está en el contexto, di exactamente: "No encuentro esa información en la documentación".
Cita siempre el documento del que sacas cada dato, entre corchetes:  [nombre_del_archivo]"""

def rag(pregunta, k=4, ver_contexto=False):
    """RAG completo = recuperar + generar respuesta."""

    trozos = recuperar(pregunta, k) #recuperamos los k chunks más relevantes para la pregunta

    contexto = "\n\n".join([f"[{fuente}]: {texto}" for texto, fuente in trozos]) #creamos un contexto con los chunks recuperados, indicando su fuente entre corchetes

    if ver_contexto:
        print("-" * 60, f"\n CONTEXTO RECUPERADO: \n\n{contexto}\n", "-" * 60)

    prompt = f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"
    return preguntar(prompt, system=SYSTEM_RAG) #mandamos el prompt al modelo de CHAT con las instrucciones del sistema    

 

In [26]:
# comparamos la MISMA pregunta con y sin RAG
print("--- SIN RAG ---")
print(preguntar(pregunta_de_prueba))
print("\n--- CON RAG ---")
print(rag(pregunta_de_prueba))

--- SIN RAG ---
Como modelo de lenguaje, no tengo acceso a tu información personal, datos de empleo, ni a las políticas internas de vacaciones de ninguna empresa específica como Lumetra.

Para saber cuántos días de vacaciones te corresponden, debes consultar:

1. **Tu departamento de Recursos Humanos (RR.HH.).**
2. **Tu contrato de trabajo.**
3. **El portal interno de la empresa (si tienen uno).**
4. **Tu supervisor directo.**

Ellos son las únicas fuentes que pueden proporcionarte esa información precisa.

--- CON RAG ---
No encuentro esa información en la documentación


In [27]:
# comparamos la MISMA pregunta con y sin RAG
print("--- SIN RAG ---")
print(preguntar(pregunta_de_prueba))
print("\n--- CON RAG ---")
print(rag("¿Cuántos días de vacaciones me corresponden al año en Lumetra?", ver_contexto=True))

--- SIN RAG ---
Como modelo de lenguaje, no tengo acceso a tu información personal, datos de empleo, ni a las políticas internas de vacaciones de ninguna empresa específica como Lumetra.

Para saber cuántos días de vacaciones te corresponden, debes consultar:

1. **Tu departamento de Recursos Humanos (RR.HH.).**
2. **Tu contrato de trabajo.**
3. **El portal interno de la empresa (si tienen uno).**
4. **Tu supervisor directo.**

Ellos son las únicas fuentes que pueden proporcionarte esa información precisa.

--- CON RAG ---
------------------------------------------------------------ 
 CONTEXTO RECUPERADO: 

[politica_teletrabajo.txt]: Lumetra funciona con un modelo híbrido 3+2: tres días de teletrabajo y dos días presenciales por semana. Los días presenciales obligatorios son martes y jueves, en la oficina de Madrid (Calle del Dato, 7, planta 3). Los equipos pueden pactar un tercer día presencial puntual para talleres o cierres de proyecto.

[manual_producto.txt]: El acuerdo de nivel d

Probad el rag()
* 3 Preguntas con respuesta -> presente en los documentos
* 1 pregunta sin respuesta
* 1 pregunta que cruce dos documentos
* ¿ Siempre cita las fuentes?


In [28]:
# Preguntas hechas por mi
preguntas = [
    "¿Qué es Lumetra?",
    "¿Qué servicios ofrece Lumetra?",
    "¿Qué problema resuelve Lumetra para sus clientes?"
]

for pregunta in preguntas:
    print("="*80)
    print("PREGUNTA:", pregunta)
    print()
    
    respuesta = rag(pregunta)
    
    print("RESPUESTA:")
    print(respuesta)

PREGUNTA: ¿Qué es Lumetra?

RESPUESTA:
Lumetra Insight es la plataforma de cuadros de mando (dashboards) de Lumetra. Se conecta a PostgreSQL, BigQuery y hojas de cálculo de Google, y refresca los datos automáticamente cada 15 minutos en todos los planes. Los dashboards se comparten por enlace o se incrustan en webs internas [manual_producto.txt].
PREGUNTA: ¿Qué servicios ofrece Lumetra?

RESPUESTA:
Lumetra ofrece los siguientes servicios:

*   Acuerdo de nivel de servicio (SLA) para los planes Pro y Enterprise, que garantiza una disponibilidad mensual del 99,9% [manual_producto.txt].
*   Lumetra Insight, la plataforma de cuadros de mando (dashboards) que se conecta a PostgreSQL, BigQuery y hojas de cálculo de Google, y refresca los datos automáticamente cada 15 minutos en todos los planes [manual_producto.txt].
*   Publicación del estado del servicio en tiempo real en status.lumetra.es, con posibilidad de suscribirse a avisos de incidencias y ventanas de mantenimiento programado [faq_s

In [29]:
# SOLUCIÓN del profesor (batería de ejemplo) — preguntas variadas para estresar al asistente:
baterias = [
    "¿Qué días tengo que ir a la oficina obligatoriamente?",
    "¿Qué incluye el plan Pro y cuánto cuesta?",
    "¿Qué hago si no me llega el correo para resetear la contraseña?",
    "¿Cuál es la política de bajas por enfermedad?",   # ¡NO está en el corpus! → debe confesarlo
    "¿Puedo teletrabajar todo julio y cuántos días seguidos de vacaciones puedo coger en agosto?",  # cruza 2 docs
]
for p in baterias:
    print(f"\n❓ {p}")
    print(rag(p))


❓ ¿Qué días tengo que ir a la oficina obligatoriamente?
Los días presenciales obligatorios son martes y jueves, en la oficina de Madrid (Calle del Dato, 7, planta 3) [politica_teletrabajo.txt].

❓ ¿Qué incluye el plan Pro y cuánto cuesta?
El plan Pro cuesta 199 euros al mes e incluye 25 usuarios, dashboards ilimitados y acceso a la API [manual_producto.txt].

❓ ¿Qué hago si no me llega el correo para resetear la contraseña?
Si no llega el correo de restablecimiento, conviene revisar la carpeta de spam y, si sigue sin aparecer, escribir a soporte indicando el correo de la cuenta [faq_soporte.txt].

❓ ¿Cuál es la política de bajas por enfermedad?
No encuentro esa información en la documentación

❓ ¿Puedo teletrabajar todo julio y cuántos días seguidos de vacaciones puedo coger en agosto?
Se puede solicitar teletrabajo total (5 días remotos por semana) durante julio y agosto. La solicitud debe presentarse en Atlas RRHH antes del 15 de junio y ser aprobada por el responsable directo [poli

## Ejercicio:

**a) El efecto de `k`.** Lanza la misma pregunta con `k=1` y `k=8`. ¿Cuándo le falta información? ¿Cuándo le sobra ruido? (con `ver_contexto=True` se ve clarísimo).

**b) Tu propio documento.** Crea `datos/politica_formacion.txt` (invéntate la política: presupuesto anual, cómo se pide…). Re-ejecuta las celdas de chunking + indexado y pregúntale por ella. Acabas de "enseñarle" algo nuevo al sistema **sin tocar el modelo**.

**c) Mini-eval.** Pasa la batería de abajo con y sin RAG y cuenta aciertos. De esta forma puedes comparar modelos y configuraciones del RAG.

**d) (Si te sobra tiempo)** Cambia el chunking: trocea por tamaño fijo (p. ej. 300 caracteres) en vez de por párrafos y repite la mini-eval. ¿Mejora o empeora? ¿Por qué?
 

In [33]:
preguntas_eval = [
    "¿Cuántos días de vacaciones anuales tengo?",
    "¿Qué días son obligatorios en oficina?",
    "¿Cuánto cuesta el plan Pro?",
    "¿En qué navegador falla la exportación a PDF?",
    "¿Qué curso es obligatorio la primera semana?",
    "¿Cuál es la política de bajas por enfermedad?",
]

In [31]:
# Con RAG
for pregunta in preguntas_eval:
    print("=" * 80)
    print("PREGUNTA:", pregunta)
    
    respuesta = rag(pregunta, k=3, ver_contexto=True)
    
    print("RESPUESTA:")
    print(respuesta)

PREGUNTA: ¿Cuántos días de vacaciones anuales tengo?
------------------------------------------------------------ 
 CONTEXTO RECUPERADO: 

[politica_teletrabajo.txt]: Durante julio y agosto se puede solicitar teletrabajo total (5 días remotos por semana). La solicitud se presenta en Atlas RRHH antes del 15 de junio y la aprueba el responsable directo. Fuera de ese periodo, el teletrabajo total solo se concede por circunstancias personales justificadas.

[politica_teletrabajo.txt]: Lumetra funciona con un modelo híbrido 3+2: tres días de teletrabajo y dos días presenciales por semana. Los días presenciales obligatorios son martes y jueves, en la oficina de Madrid (Calle del Dato, 7, planta 3). Los equipos pueden pactar un tercer día presencial puntual para talleres o cierres de proyecto.

[onboarding.txt]: Cada nueva incorporación tiene asignado un buddy durante las primeras 6 semanas: una persona del equipo que resuelve las dudas del día a día y hace las presentaciones informales. El b

In [34]:
# Con k=1 vs k=8
for pregunta in preguntas_eval:
    print("=" * 80)
    print("PREGUNTA:", pregunta)

    print("\n--- k=1 ---")
    print(rag(pregunta, k=1, ver_contexto=True))

    print("\n--- k=8 ---")
    print(rag(pregunta, k=8, ver_contexto=True))

PREGUNTA: ¿Cuántos días de vacaciones anuales tengo?

--- k=1 ---
------------------------------------------------------------ 
 CONTEXTO RECUPERADO: 

[politica_teletrabajo.txt]: Durante julio y agosto se puede solicitar teletrabajo total (5 días remotos por semana). La solicitud se presenta en Atlas RRHH antes del 15 de junio y la aprueba el responsable directo. Fuera de ese periodo, el teletrabajo total solo se concede por circunstancias personales justificadas.
 ------------------------------------------------------------
No encuentro esa información en la documentación

--- k=8 ---
------------------------------------------------------------ 
 CONTEXTO RECUPERADO: 

[politica_teletrabajo.txt]: Durante julio y agosto se puede solicitar teletrabajo total (5 días remotos por semana). La solicitud se presenta en Atlas RRHH antes del 15 de junio y la aprueba el responsable directo. Fuera de ese periodo, el teletrabajo total solo se concede por circunstancias personales justificadas.

[

In [35]:
from pathlib import Path

ruta = Path("datos/politica_formacion.txt")

texto = ruta.read_text(encoding="utf-8")

print(texto)

In [36]:
chunks = []

for i, parrafo in enumerate(texto.split("\n\n")):
    parrafo = parrafo.strip()

    if parrafo:
        chunks.append(parrafo)

print("Número de chunks:", len(chunks))

Número de chunks: 0
